# Phase 03B: Teacher A ExpansionNet v2 Offline Preflight

Notebook này **thay Teacher A CLIP ViT-B/32** bằng checkpoint chính thức
**ExpansionNet v2 single-model** trong cùng pipeline . Output CLIP cũ
được giữ nguyên như bằng chứng lịch sử, nhưng không được dùng làm input cho
luồng Teacher A mới.

## Quyết định phương pháp đã khóa

- Student ghép cặp vẫn là **S1 MobileNetV3-Large**.
- ExpansionNet v2 dùng decoder/vocabulary native 10.000 từ.
- S1 dùng shared decoder/vocabulary Phase 1 gồm 10.080 từ.
- Vì token space khác nhau, **direct Logit KD bị cấm**.
- Cơ chế reproduction hợp lệ là **Sequence KD**: Teacher sinh pseudo-caption,
  sau đó S1 học bằng CE trên chuỗi đã ánh xạ lại qua tokenizer Phase 1.
- Feature KD chỉ được phép ở Phase 5 sau khi định nghĩa projection/alignment
  riêng; không được kế thừa identity projection của CLIP.

## Kaggle inputs

1. Phase 1 Formal Output.
2. Phase 2A Shared Student Pipeline.
3. Phase 3A Teacher Pool Entry Gate.
4. MS COCO 2014 images.
5. Offline package `expansionnet_v2_ieee_bigdata2023`.
6. Offline wheel `pycocoevalcap==1.2` nếu runtime chưa cài sẵn.

GPU T4 hoặc mạnh hơn, Internet Off. Chạy tuần tự từng cell.
Held-out Test luôn `SEALED`.


## Cell 1 — Environment, deterministic policy và input discovery

**Mục tiêu:** tạo output riêng cho ExpansionNet v2, nhận GPU, tìm đúng Phase
1/2A/3A và offline package; chưa đọc Parquet, ảnh hay checkpoint.

**Đầu ra dự kiến:** `PASSED`, tiến độ `1/7`, Test `SEALED`.


In [1]:
import csv
import gc
import hashlib
import json
import math
import os
import pickle
import platform
import random
import subprocess
import sys
import time
from argparse import Namespace
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch


CELL_STARTED_AT = time.perf_counter()
SEED = 42
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path(
    "/kaggle/working//"
    "phase3_teacher_a_expansionnet_v2_preflight"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
VALIDATION_DIR = OUTPUT_ROOT / "validation"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT, AUDIT_DIR, CONFIG_DIR, VALIDATION_DIR, MANIFEST_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()


def set_deterministic(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.use_deterministic_algorithms(True, warn_only=False)
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)


def find_expansionnet_manifest():
    matches = []
    for candidate in INPUT_ROOT.rglob("MODEL_SOURCE_MANIFEST.json"):
        try:
            payload = read_json(candidate)
        except Exception:
            continue
        if payload.get("model_id") == "expansionnet_v2":
            matches.append(candidate.resolve())
    assert len(matches) == 1, (
        "Cần gắn đúng một offline ExpansionNet v2 package; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0]


if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA không khả dụng. Trong Kaggle chọn GPU T4 hoặc mạnh hơn, "
        "khởi động session mới rồi chạy lại Cell 1."
    )
set_deterministic(SEED)
device = torch.device("cuda")

PHASE1_READY = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
PHASE2_READY = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
PHASE3A_READY = exactly_one("PHASE3_TEACHER_POOL_INPUT_READY.json")
MODEL_MANIFEST_PATH = find_expansionnet_manifest()
MODEL_ROOT = MODEL_MANIFEST_PATH.parent

write_json(
    AUDIT_DIR / "phase3b_cell01_environment.json",
    {
        "audit_version": "phase3b_expansionnet_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "strict_deterministic": True,
        "phase1_marker": str(PHASE1_READY),
        "phase2_marker": str(PHASE2_READY),
        "phase3a_marker": str(PHASE3A_READY),
        "model_manifest": str(MODEL_MANIFEST_PATH),
        "dataset_read": False,
        "checkpoint_payload_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03B EXPANSIONNET V2 CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher                   : A — ExpansionNet v2")
print("Matched Student           : S1 — MobileNetV3-Large")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Phase 1 / 2A / 3A         : FOUND / FOUND / FOUND")
print("Offline model package     : FOUND")
print("Strict deterministic      : True")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 7 code cells")
print("Next                      : verify provenance, hashes and KD compatibility")


PHASE 03B EXPANSIONNET V2 CELL 1 STATUS
Status                    : PASSED
Teacher                   : A — ExpansionNet v2
Matched Student           : S1 — MobileNetV3-Large
GPU                       : Tesla T4
Phase 1 / 2A / 3A         : FOUND / FOUND / FOUND
Offline model package     : FOUND
Strict deterministic      : True
Test Set status           : SEALED
Progress                  : 1 / 7 code cells
Next                      : verify provenance, hashes and KD compatibility


## Cell 2 — Provenance, checksum, vocabulary và KD compatibility

**Mục tiêu:** xác minh checkpoint/source theo manifest; đọc hai vocabulary để
chứng minh token space không tương thích.

**Gate bắt buộc:** `direct_logit_kd_allowed=False`,
`sequence_kd_required=True`.


In [2]:
CELL_STARTED_AT = time.perf_counter()

model_manifest = read_json(MODEL_MANIFEST_PATH)
assert model_manifest["schema_version"] == (
    "expansionnet_v2_offline_bundle_v1"
)
assert model_manifest["teacher_id"] == "A"
assert model_manifest["upstream_commit"] == (
    "eb7f1c98d00cdaa61b60c43852155b3742fe5b45"
)
assert model_manifest["publication"] == (
    "2023 IEEE International Conference on Big Data"
)
assert model_manifest["doi"] == "10.1109/BigData59044.2023.10386812"
assert model_manifest["license"] == "MIT"


def verify_record(record):
    target = (MODEL_ROOT / record["relative_path"]).resolve()
    assert target.is_relative_to(MODEL_ROOT.resolve())
    assert target.is_file(), target
    assert target.stat().st_size == int(record["size_bytes"])
    assert sha256_file(target) == record["sha256"]
    return target


CHECKPOINT_PATH = verify_record(model_manifest["checkpoint"])
NATIVE_VOCAB_PATH = verify_record(model_manifest["vocabulary"])
for record in model_manifest["required_files"]:
    verify_record(record)
for record in model_manifest["integration_patches"]:
    target = (MODEL_ROOT / record["file"]).resolve()
    assert target.is_file()
    assert sha256_file(target) == record["sha256"]

with NATIVE_VOCAB_PATH.open("rb") as handle:
    NATIVE_TOKENS = pickle.load(handle)
NATIVE_WORD2IDX = NATIVE_TOKENS["word2idx_dict"]
NATIVE_IDX2WORD = NATIVE_TOKENS["idx2word_list"]
NATIVE_BOS = int(NATIVE_WORD2IDX[NATIVE_TOKENS["sos_str"]])
NATIVE_EOS = int(NATIVE_WORD2IDX[NATIVE_TOKENS["eos_str"]])
NATIVE_PAD = int(NATIVE_WORD2IDX["PAD"])
NATIVE_UNK = int(NATIVE_WORD2IDX["UNK"])
assert len(NATIVE_WORD2IDX) == 10000
assert (NATIVE_PAD, NATIVE_BOS, NATIVE_EOS, NATIVE_UNK) == (
    78, 79, 77, 80
)

PHASE1_VOCAB_PATH = exactly_one("word_vocabulary_minfreq5.json")
phase1_vocabulary = read_json(PHASE1_VOCAB_PATH)
assert int(phase1_vocabulary["vocabulary_size"]) == 10080
special = phase1_vocabulary["special_token_ids"]
assert (
    int(special["<pad>"]),
    int(special["<bos>"]),
    int(special["<eos>"]),
    int(special["<unk>"]),
) == (0, 1, 2, 3)

TOKEN_SPACES_COMPATIBLE = False
DIRECT_LOGIT_KD_ALLOWED = False
SEQUENCE_KD_REQUIRED = True
FEATURE_KD_REQUIRES_PROJECTION = True

compatibility_contract = {
    "contract_version": "teacher_a_expansionnet_v2_kd_compatibility_v1",
    "created_at_utc": now_utc(),
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "teacher_vocabulary_size": 10000,
    "teacher_special_ids": {
        "pad": NATIVE_PAD, "bos": NATIVE_BOS,
        "eos": NATIVE_EOS, "unk": NATIVE_UNK,
    },
    "student_vocabulary_size": 10080,
    "student_special_ids": {"pad": 0, "bos": 1, "eos": 2, "unk": 3},
    "token_spaces_compatible": False,
    "direct_logit_kd_allowed": False,
    "sequence_kd_required": True,
    "sequence_kd_mapping": (
        "teacher token IDs -> words -> Phase 1 tokenizer -> student token IDs"
    ),
    "feature_kd_requires_projection": True,
    "old_clip_feature_identity_projection_allowed": False,
    "test_set_status": "sealed",
}
COMPATIBILITY_PATH = (
    CONFIG_DIR / "teacher_a_expansionnet_v2_kd_compatibility_contract.json"
)
write_json(COMPATIBILITY_PATH, compatibility_contract)
write_json(
    AUDIT_DIR / "phase3b_cell02_source_and_compatibility.json",
    {
        "audit_version": "phase3b_expansionnet_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "checkpoint_size_bytes": CHECKPOINT_PATH.stat().st_size,
        "checkpoint_sha256": sha256_file(CHECKPOINT_PATH),
        "source_commit": model_manifest["upstream_commit"],
        "native_vocabulary_verified": True,
        "phase1_vocabulary_verified": True,
        "token_spaces_compatible": False,
        "direct_logit_kd_allowed": False,
        "sequence_kd_required": True,
        "network_download": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03B EXPANSIONNET V2 CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Official checkpoint       : VERIFIED")
print("Checkpoint SHA-256        :", sha256_file(CHECKPOINT_PATH))
print("Source commit             :", model_manifest["upstream_commit"])
print("Teacher vocabulary        : 10000 | PAD/BOS/EOS/UNK = 78/79/77/80")
print("Student vocabulary        : 10080 | PAD/BOS/EOS/UNK = 0/1/2/3")
print("Token spaces compatible   : False")
print("Direct Logit KD allowed   : False")
print("Sequence KD required      : True")
print("Feature projection needed : True")
print("Network download          : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 7 code cells")
print("Next                      : build Validation-only dataset")


PHASE 03B EXPANSIONNET V2 CELL 2 STATUS
Status                    : PASSED
Official checkpoint       : VERIFIED
Checkpoint SHA-256        : 2725a827ffbb0b2d5cf0f9c39ce0a9db5f1a93ec3017158f0edb08aab7ace4f0
Source commit             : eb7f1c98d00cdaa61b60c43852155b3742fe5b45
Teacher vocabulary        : 10000 | PAD/BOS/EOS/UNK = 78/79/77/80
Student vocabulary        : 10080 | PAD/BOS/EOS/UNK = 0/1/2/3
Token spaces compatible   : False
Direct Logit KD allowed   : False
Sequence KD required      : True
Feature projection needed : True
Network download          : False
Test Set status           : SEALED
Progress                  : 2 / 7 code cells
Next                      : build Validation-only dataset


## Cell 3 — Validation-only dataset và preprocessing

**Mục tiêu:** dựng đúng 5.000 ảnh Karpathy Validation và 25.010 reference;
không mở Held-out Test Parquet/ảnh. Khóa preprocessing RGB 384×384,
bilinear và ImageNet mean/std đúng source chính thức.


In [3]:
CELL_STARTED_AT = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

IMAGE_MANIFEST_PATH = exactly_one("mscoco_image_manifest.parquet")
VALIDATION_PARQUET_PATH = exactly_one("mscoco_validation.parquet")


def find_image_leaf(name, prefix):
    matches = []
    for path in INPUT_ROOT.rglob(name):
        if path.is_dir() and next(path.glob(f"{prefix}*.jpg"), None):
            matches.append(path.resolve())
    matches = sorted(set(matches))
    assert len(matches) == 1, matches
    return matches[0]


TRAIN_LEAF = find_image_leaf("train2014", "COCO_train2014_")
VAL_LEAF = find_image_leaf("val2014", "COCO_val2014_")

manifest_table = pq.read_table(
    IMAGE_MANIFEST_PATH,
    columns=[
        "image_id", "experiment_split", "source_partition", "image_path"
    ],
)
VALIDATION_PATHS = {}
HELD_OUT_TEST_IDS = set()
for row in manifest_table.to_pylist():
    image_id = int(row["image_id"])
    split = row["experiment_split"]
    if split == "held_out_test":
        HELD_OUT_TEST_IDS.add(image_id)
        continue
    if split != "validation":
        continue
    leaf = (
        TRAIN_LEAF
        if row["source_partition"] == "train2014"
        else VAL_LEAF
    )
    VALIDATION_PATHS[image_id] = (
        leaf / Path(row["image_path"]).name
    )
assert len(VALIDATION_PATHS) == 5000
assert len(HELD_OUT_TEST_IDS) == 5000
assert set(VALIDATION_PATHS).isdisjoint(HELD_OUT_TEST_IDS)

VALIDATION_REFERENCES = defaultdict(list)
for batch in pq.ParquetFile(VALIDATION_PARQUET_PATH).iter_batches(
    batch_size=8192, columns=["image_id", "caption"]
):
    for row in batch.to_pylist():
        VALIDATION_REFERENCES[int(row["image_id"])].append(row["caption"])
assert len(VALIDATION_REFERENCES) == 5000
assert sum(map(len, VALIDATION_REFERENCES.values())) == 25010
assert set(VALIDATION_REFERENCES) == set(VALIDATION_PATHS)
assert set(VALIDATION_REFERENCES).isdisjoint(HELD_OUT_TEST_IDS)

VALIDATION_IMAGE_IDS = sorted(VALIDATION_PATHS)
PREPROCESS = transforms.Compose(
    [
        transforms.Resize(
            (384, 384), interpolation=InterpolationMode.BILINEAR
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)


class ValidationImages(Dataset):
    def __len__(self):
        return len(VALIDATION_IMAGE_IDS)

    def __getitem__(self, index):
        image_id = VALIDATION_IMAGE_IDS[index]
        path = VALIDATION_PATHS[image_id]
        with Image.open(path) as source:
            image = PREPROCESS(source.convert("RGB"))
        return {"image_id": image_id, "image": image}


VALIDATION_DATASET = ValidationImages()
VALIDATION_LOADER = DataLoader(
    VALIDATION_DATASET,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
)
sample_a = VALIDATION_DATASET[0]["image"]
sample_b = VALIDATION_DATASET[0]["image"]
assert sample_a.shape == (3, 384, 384)
assert torch.isfinite(sample_a).all()
assert torch.equal(sample_a, sample_b)
for image_id in VALIDATION_IMAGE_IDS[:256]:
    assert VALIDATION_PATHS[image_id].is_file()

REFERENCES_PATH = VALIDATION_DIR / "karpathy_validation_references.json"
write_json(
    REFERENCES_PATH,
    {
        str(image_id): captions
        for image_id, captions in VALIDATION_REFERENCES.items()
    },
)
write_json(
    CONFIG_DIR / "teacher_a_expansionnet_v2_preprocessing_contract.json",
    {
        "contract_version": "teacher_a_expansionnet_v2_preprocess_v1",
        "input": "RGB float32 [3,384,384]",
        "resize": [384, 384],
        "interpolation": "bilinear",
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
        "validation_deterministic": True,
        "test_set_status": "sealed",
    },
)
write_json(
    AUDIT_DIR / "phase3b_cell03_validation_data.json",
    {
        "audit_version": "phase3b_expansionnet_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_images": 5000,
        "validation_references": 25010,
        "held_out_test_ids": 5000,
        "validation_test_overlap": 0,
        "image_probe_passed": 256,
        "preprocessing_repeat_exact": True,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03B EXPANSIONNET V2 CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
print("Validation/Test overlap   : 0")
print("Input tensor              : RGB [3,384,384] float32")
print("Normalization             : ImageNet mean/std")
print("Repeat exact              : True")
print("Test Parquet opened       : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 7 code cells")
print("Next                      : reconstruct and strict-load model")


PHASE 03B EXPANSIONNET V2 CELL 3 STATUS
Status                    : PASSED
Validation images         : 5000
Validation references     : 25010
Validation/Test overlap   : 0
Input tensor              : RGB [3,384,384] float32
Normalization             : ImageNet mean/std
Repeat exact              : True
Test Parquet opened       : False
Test images read          : False
Test Set status           : SEALED
Progress                  : 3 / 7 code cells
Next                      : reconstruct and strict-load model


## Cell 4 — Reconstruct native ExpansionNet v2 và strict-load

**Mục tiêu:** dựng đúng Swin-L + ExpansionNet v2 decoder native, strict-load
checkpoint chính thức, không tạo optimizer và không huấn luyện.


In [4]:
CELL_STARTED_AT = time.perf_counter()

SOURCE_ROOT = MODEL_ROOT / "ExpansionNet_v2_src"
assert SOURCE_ROOT.is_dir()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from models.End_ExpansionNet_v2 import End_ExpansionNet_v2

drop_args = Namespace(
    enc=0.0, dec=0.0, enc_input=0.0, dec_input=0.0, other=0.0
)
TEACHER = End_ExpansionNet_v2(
    swin_img_size=384,
    swin_patch_size=4,
    swin_in_chans=3,
    swin_embed_dim=192,
    swin_depths=[2, 2, 18, 2],
    swin_num_heads=[6, 12, 24, 48],
    swin_window_size=12,
    swin_mlp_ratio=4.0,
    swin_qkv_bias=True,
    swin_qk_scale=None,
    swin_drop_rate=0.0,
    swin_attn_drop_rate=0.0,
    swin_drop_path_rate=0.0,
    swin_norm_layer=torch.nn.LayerNorm,
    swin_ape=False,
    swin_patch_norm=True,
    swin_use_checkpoint=False,
    final_swin_dim=1536,
    d_model=512,
    N_enc=3,
    N_dec=3,
    num_heads=8,
    ff=2048,
    num_exp_enc_list=[32, 64, 128, 256, 512],
    num_exp_dec=16,
    output_word2idx=NATIVE_WORD2IDX,
    output_idx2word=NATIVE_IDX2WORD,
    max_seq_len=74,
    drop_args=drop_args,
    rank=device,
)
checkpoint = torch.load(
    CHECKPOINT_PATH, map_location="cpu", weights_only=False
)
assert "model_state_dict" in checkpoint
incompatible = TEACHER.load_state_dict(
    checkpoint["model_state_dict"], strict=True
)
assert not incompatible.missing_keys
assert not incompatible.unexpected_keys
del checkpoint
gc.collect()

TEACHER = TEACHER.to(device).eval()
for parameter in TEACHER.parameters():
    parameter.requires_grad_(False)
PARAMETERS = sum(parameter.numel() for parameter in TEACHER.parameters())
TRAINABLE_PARAMETERS = sum(
    parameter.numel()
    for parameter in TEACHER.parameters()
    if parameter.requires_grad
)
assert TRAINABLE_PARAMETERS == 0
assert TEACHER.training is False

architecture_contract = {
    "contract_version": "teacher_a_expansionnet_v2_architecture_v1",
    "created_at_utc": now_utc(),
    "teacher": "ExpansionNet v2 single model",
    "backbone": "Swin-L patch4 window12 384",
    "image_size": 384,
    "model_dim": 512,
    "encoder_layers": 3,
    "decoder_layers": 3,
    "attention_heads": 8,
    "feedforward_dim": 2048,
    "native_vocabulary_size": 10000,
    "native_max_sequence_length": 74,
    "parameters": PARAMETERS,
    "checkpoint_strict_load": True,
    "missing_keys": [],
    "unexpected_keys": [],
    "frozen_teacher": True,
    "optimizer_created": False,
    "training_started": False,
    "test_set_status": "sealed",
}
ARCHITECTURE_PATH = (
    CONFIG_DIR / "teacher_a_expansionnet_v2_architecture_contract.json"
)
write_json(ARCHITECTURE_PATH, architecture_contract)
write_json(
    AUDIT_DIR / "phase3b_cell04_model_load.json",
    {
        "audit_version": "phase3b_expansionnet_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "parameters": PARAMETERS,
        "trainable_parameters": TRAINABLE_PARAMETERS,
        "strict_load": True,
        "missing_keys": 0,
        "unexpected_keys": 0,
        "optimizer_created": False,
        "optimizer_step_performed": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03B EXPANSIONNET V2 CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Backbone                  : Swin-L patch4 window12 384")
print("Encoder / decoder layers  : 3 / 3")
print("Model dim / heads         : 512 / 8")
print("Native vocabulary         : 10000")
print("Parameters                :", PARAMETERS)
print("Strict checkpoint load    : PASSED")
print("Missing/unexpected keys   : 0 / 0")
print("Frozen teacher            : True")
print("Optimizer created         : False")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 7 code cells")
print("Next                      : deterministic Beam-3 smoke")


PHASE 03B EXPANSIONNET V2 CELL 4 STATUS
Status                    : PASSED
Backbone                  : Swin-L patch4 window12 384
Encoder / decoder layers  : 3 / 3
Model dim / heads         : 512 / 8
Native vocabulary         : 10000
Parameters                : 233803076
Strict checkpoint load    : PASSED
Missing/unexpected keys   : 0 / 0
Frozen teacher            : True
Optimizer created         : False
Training started          : False
Test Set status           : SEALED
Progress                  : 4 / 7 code cells
Next                      : deterministic Beam-3 smoke


## Cell 5 — Deterministic Beam-3 smoke và resource profile

**Mục tiêu:** sinh caption cho hai ảnh hai lần, kiểm token/caption/score lặp
chính xác, finite tensors và đo peak GPU memory. Đây chưa phải full Validation.


In [5]:
CELL_STARTED_AT = time.perf_counter()


def decode_native(tokens):
    words = []
    for token in tokens:
        token = int(token)
        if token == NATIVE_BOS:
            continue
        if token == NATIVE_EOS:
            break
        if token == NATIVE_PAD:
            continue
        words.append(str(NATIVE_IDX2WORD[token]))
    return " ".join(words).strip()


def generate_batch(images):
    images = images.to(device, non_blocking=True)
    with torch.inference_mode():
        predictions, log_probs = TEACHER(
            enc_x=images,
            enc_x_num_pads=[0] * images.shape[0],
            mode="beam_search",
            beam_size=3,
            beam_max_seq_len=31,
            sample_or_max="max",
            how_many_outputs=1,
            sos_idx=NATIVE_BOS,
            eos_idx=NATIVE_EOS,
            length_penalty=0.7,
        )
    tokens = [
        [int(value) for value in predictions[index][0]]
        for index in range(len(predictions))
    ]
    captions = [decode_native(values) for values in tokens]
    if torch.is_tensor(log_probs):
        scores = log_probs.detach().cpu()
    else:
        scores = log_probs
    return tokens, captions, scores


smoke_samples = [VALIDATION_DATASET[index] for index in range(2)]
smoke_images = torch.stack([item["image"] for item in smoke_samples])
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
tokens_a, captions_a, scores_a = generate_batch(smoke_images)
tokens_b, captions_b, scores_b = generate_batch(smoke_images)
assert tokens_a == tokens_b
assert captions_a == captions_b
assert all(caption for caption in captions_a)
if torch.is_tensor(scores_a):
    assert torch.equal(scores_a, scores_b)
    assert torch.isfinite(scores_a).all()
else:
    assert scores_a == scores_b
peak_memory_mib = torch.cuda.max_memory_allocated() / (1024 ** 2)

smoke_payload = []
for index, sample in enumerate(smoke_samples):
    smoke_payload.append(
        {
            "image_id": int(sample["image_id"]),
            "token_ids": tokens_a[index],
            "caption": captions_a[index],
        }
    )
write_json(VALIDATION_DIR / "beam3_smoke_predictions.json", smoke_payload)
write_json(
    AUDIT_DIR / "phase3b_cell05_beam_smoke.json",
    {
        "audit_version": "phase3b_expansionnet_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "images": 2,
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_decoder_steps": 31,
        "token_ids_repeat_exact": True,
        "captions_repeat_exact": True,
        "scores_repeat_exact": True,
        "captions": captions_a,
        "peak_gpu_memory_mib": peak_memory_mib,
        "full_validation_run": False,
        "optimizer_created": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03B EXPANSIONNET V2 CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Beam size / penalty       : 3 / 0.7")
print("Maximum decoder steps     : 31")
print("Token IDs repeat exact    : True")
print("Captions repeat exact     : True")
print("Scores repeat exact       : True")
for item in smoke_payload:
    print(f'image_id={item["image_id"]} | caption={item["caption"]}')
print("Peak GPU memory           :", round(peak_memory_mib, 2), "MiB")
print("Full Validation run       : False")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 7 code cells")
print("Next                      : full 5000-image Validation quality gate")


/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:206: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:347.)
  new_loop_dec_classes = loop_dec_classes[[bs_idxes, which_sequence]]
/kaggle/input/models/nguynthnhdan/expansionnet-v2-ieee-bigdata2023/pytorch/default/1/expansionnet_v2_ieee_bigdata2023/ExpansionNet_v2_src/models/captioning_model.py:207: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor

PHASE 03B EXPANSIONNET V2 CELL 5 STATUS
Status                    : PASSED
Beam size / penalty       : 3 / 0.7
Maximum decoder steps     : 31
Token IDs repeat exact    : True
Captions repeat exact     : True
Scores repeat exact       : True
image_id=139 | caption=a living room with a table and a television
image_id=285 | caption=a large brown bear standing in the grass
Peak GPU memory           : 1167.33 MiB
Full Validation run       : False
Test Set status           : SEALED
Progress                  : 5 / 7 code cells
Next                      : full 5000-image Validation quality gate


## Cell 6 — Full Karpathy Validation quality gate

Cell này sinh caption cho đủ **5.000 ảnh Validation**, tính BLEU-1, BLEU-4,
METEOR, ROUGE-L và CIDEr-D bằng `pycocoevalcap==1.2`.

Checkpoint chỉ được chấp nhận nếu CIDEr-D lớn hơn mức trung bình Teacher A
CLIP cũ `1.1604642792`. CE không được so trực tiếp vì hai vocabulary khác nhau.

**Lưu ý:** đây là cell lâu nhất; có ghi predictions định kỳ để theo dõi.


In [6]:
CELL_STARTED_AT = time.perf_counter()
try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer


def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(tokenized_gt, tokenized_pred)
    meteor, _ = Meteor().compute_score(tokenized_gt, tokenized_pred)
    rouge, _ = Rouge().compute_score(tokenized_gt, tokenized_pred)
    cider, _ = Cider().compute_score(tokenized_gt, tokenized_pred)
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }


PREDICTIONS_PATH = (
    VALIDATION_DIR / "expansionnet_v2_validation_predictions.json"
)
hypotheses = {}
generation_started = time.perf_counter()
for batch_index, batch in enumerate(VALIDATION_LOADER):
    image_id = int(batch["image_id"][0])
    tokens, captions, _ = generate_batch(batch["image"])
    hypotheses[image_id] = captions[0]
    if (batch_index + 1) % 250 == 0:
        write_json(
            PREDICTIONS_PATH,
            {
                str(key): value
                for key, value in sorted(hypotheses.items())
            },
        )
        print(
            f"Validation generation: {batch_index + 1}/5000 | "
            f"elapsed={time.perf_counter() - generation_started:.1f}s"
        )

assert len(hypotheses) == 5000
assert set(hypotheses) == set(VALIDATION_REFERENCES)
assert all(hypotheses.values())
write_json(
    PREDICTIONS_PATH,
    {
        str(key): value
        for key, value in sorted(hypotheses.items())
    },
)
metrics = compute_coco_metrics(VALIDATION_REFERENCES, hypotheses)
assert all(math.isfinite(value) for value in metrics.values())

OLD_CLIP_CIDER_MEAN = 1.1604642791728237
QUALITY_GATE_PASSED = metrics["CIDEr-D"] > OLD_CLIP_CIDER_MEAN
assert QUALITY_GATE_PASSED, (
    "ExpansionNet v2 không vượt Teacher A CLIP cũ trên cùng Validation: "
    f'{metrics["CIDEr-D"]:.6f} <= {OLD_CLIP_CIDER_MEAN:.6f}. '
    "Không chuyển Phase 03C; kiểm tra preprocessing/checkpoint."
)
METRICS_PATH = (
    VALIDATION_DIR / "expansionnet_v2_validation_metrics.json"
)
write_json(
    METRICS_PATH,
    {
        "metric_backend": "pycocoevalcap 1.2",
        "split": "Karpathy Validation",
        "images": 5000,
        "references": 25010,
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_decoder_steps": 31,
        "metrics": metrics,
        "old_clip_teacher_a_cider_d_mean": OLD_CLIP_CIDER_MEAN,
        "cider_d_delta_vs_old_clip": (
            metrics["CIDEr-D"] - OLD_CLIP_CIDER_MEAN
        ),
        "quality_gate_passed": True,
        "teacher_ce_comparable_to_student": False,
        "test_set_status": "sealed",
    },
)
write_json(
    AUDIT_DIR / "phase3b_cell06_full_validation.json",
    {
        "audit_version": "phase3b_expansionnet_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "validation_images": 5000,
        "validation_references": 25010,
        "metrics": metrics,
        "quality_gate": (
            "CIDEr-D > old CLIP Teacher A three-seed mean"
        ),
        "quality_gate_passed": True,
        "teacher_ce_reported": False,
        "reason_ce_not_reported": (
            "native ExpansionNet vocabulary differs from Phase 1 vocabulary"
        ),
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_images_read": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03B EXPANSIONNET V2 CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Validation images         : 5000")
print("Validation references     : 25010")
for key, value in metrics.items():
    print(f"{key:<26}: {value:.6f}")
print("Old CLIP CIDEr-D mean     :", f"{OLD_CLIP_CIDER_MEAN:.6f}")
print(
    "CIDEr-D delta            :",
    f'{metrics["CIDEr-D"] - OLD_CLIP_CIDER_MEAN:+.6f}',
)
print("Quality gate              : PASSED")
print("Teacher CE comparable     : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 7 code cells")
print("Next                      : contracts, manifest and READY marker")


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl
Validation generation: 250/5000 | elapsed=47.0s
Validation generation: 500/5000 | elapsed=94.7s
Validation generation: 750/5000 | elapsed=141.7s
Validation generation: 1000/5000 | elapsed=188.8s
Validation generation: 1250/5000 | elapsed=235.9s
Validation generation: 1500/5000 | elapsed=282.8s
Validation generation: 1750/5000 | elapsed=330.2s
Validation generation: 2000/5000 | elapsed=377.7s
Validation generation: 2250/5000 | elapsed=425.4s
Validation generation: 2500/5000 | elapsed=472.5s
Validation generation: 2750/5000 | elapsed=519.5s
Validation generation: 3000/5000 | elapsed=566.5s
Validation generation: 3250/5000 | elapsed=613.1s
Validation generation: 3500/5000 | elapsed=660.2s
Validation generation: 3750/5000 | elapsed=707.0s
Validation generation: 4000/5000 | elapsed=753.8s
Validation generation: 4250/5000 | elapsed=800.2s
Validation generation: 4500/5000 | elapsed=847.5s
Valid

PTBTokenizer tokenized 307821 tokens at 510103.51 tokens per second.
PTBTokenizer tokenized 52644 tokens at 296502.56 tokens per second.


{'testlen': 47645, 'reflen': 47632, 'guess': [47645, 42645, 37645, 32645], 'correct': [39650, 24188, 12635, 6169]}
ratio: 1.0002729257641712
PHASE 03B EXPANSIONNET V2 CELL 6 STATUS
Status                    : PASSED
Validation images         : 5000
Validation references     : 25010
BLEU-1                    : 0.832196
BLEU-4                    : 0.415964
METEOR                    : 0.303835
ROUGE-L                   : 0.607139
CIDEr-D                   : 1.397207
Old CLIP CIDEr-D mean     : 1.160464
CIDEr-D delta            : +0.236743
Quality gate              : PASSED
Teacher CE comparable     : False
Test metrics computed     : False
Test Set status           : SEALED
Progress                  : 6 / 7 code cells
Next                      : contracts, manifest and READY marker


## Cell 7 — Final contracts, manifest và Phase 03C authorization

Cell cuối chỉ tạo marker khi toàn bộ checksum, deterministic smoke, full
Validation và KD compatibility gate đều đã PASS.


In [7]:
CELL_STARTED_AT = time.perf_counter()

assert DIRECT_LOGIT_KD_ALLOWED is False
assert SEQUENCE_KD_REQUIRED is True
assert FEATURE_KD_REQUIRES_PROJECTION is True
assert QUALITY_GATE_PASSED is True
assert len(hypotheses) == 5000

sequence_kd_contract = {
    "contract_version": "teacher_a_s1_sequence_kd_reproduction_v1",
    "created_at_utc": now_utc(),
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "teacher_mode": "frozen eval inference",
    "pseudo_caption_split": "Train+Restval only",
    "validation_selection_split": "Karpathy Validation only",
    "teacher_generation": {
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_decoder_steps": 31,
        "native_vocabulary_size": 10000,
    },
    "student_target_conversion": (
        "native teacher IDs -> words -> locked Phase 1 tokenizer -> "
        "student IDs with BOS/EOS"
    ),
    "student_vocabulary_size": 10080,
    "loss": "student token cross-entropy on teacher pseudo-caption",
    "direct_logit_kd_allowed": False,
    "feature_kd_in_phase4_allowed": False,
    "feature_kd_requires_new_phase5_projection": True,
    "test_set_status": "sealed",
}
SEQUENCE_KD_PATH = (
    CONFIG_DIR / "teacher_a_s1_sequence_kd_reproduction_contract.json"
)
write_json(SEQUENCE_KD_PATH, sequence_kd_contract)

artifact_paths = sorted(
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file()
    and path.name not in {
        "phase3b_expansionnet_v2_manifest.csv",
        "SHA256SUMS",
        "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json",
    }
)
MANIFEST_CSV = MANIFEST_DIR / "phase3b_expansionnet_v2_manifest.csv"
with MANIFEST_CSV.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    for artifact in artifact_paths:
        writer.writerow(
            {
                "relative_path": artifact.relative_to(OUTPUT_ROOT).as_posix(),
                "size_bytes": artifact.stat().st_size,
                "sha256": sha256_file(artifact),
            }
        )

manifest_rows = list(
    csv.DictReader(
        MANIFEST_CSV.open("r", encoding="utf-8", newline="")
    )
)
SHA256SUMS_PATH = MANIFEST_DIR / "SHA256SUMS"
SHA256SUMS_PATH.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)
for row in manifest_rows:
    artifact = OUTPUT_ROOT / row["relative_path"]
    assert artifact.is_file()
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

READY_PATH = (
    OUTPUT_ROOT
    / "PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json"
)
ready_payload = {
    "schema_version": "phase3_teacher_a_expansionnet_v2_preflight_ready_v1",
    "created_at_utc": now_utc(),
    "status": "ready",
    "project_version": "",
    "phase": "03B",
    "teacher_id": "A",
    "teacher": "ExpansionNet v2",
    "replaces": "CLIP ViT-B/32 + shared decoder",
    "matched_student": "S1 MobileNetV3-Large",
    "checkpoint_sha256": sha256_file(CHECKPOINT_PATH),
    "checkpoint_strict_load": True,
    "validation_metrics": metrics,
    "quality_gate_passed": True,
    "direct_logit_kd_allowed": False,
    "sequence_kd_required": True,
    "feature_kd_requires_projection": True,
    "architecture_contract": {
        "relative_path": ARCHITECTURE_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(ARCHITECTURE_PATH),
    },
    "compatibility_contract": {
        "relative_path": COMPATIBILITY_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(COMPATIBILITY_PATH),
    },
    "sequence_kd_contract": {
        "relative_path": SEQUENCE_KD_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(SEQUENCE_KD_PATH),
    },
    "manifest": {
        "csv": {
            "relative_path": MANIFEST_CSV.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MANIFEST_CSV),
            "rows": len(manifest_rows),
        },
        "sha256sums": {
            "relative_path": SHA256SUMS_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(SHA256SUMS_PATH),
        },
    },
    "phase03c_authorized": True,
    "full_training_started": False,
    "test_parquet_opened": False,
    "test_caption_read": False,
    "test_images_read": False,
    "test_metrics_computed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json(READY_PATH, ready_payload)

print("=" * 72)
print("PHASE 03B EXPANSIONNET V2 FINAL STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher                   : A — ExpansionNet v2")
print("Matched Student           : S1 — MobileNetV3-Large")
print("Checkpoint strict load    : PASSED")
print("Validation CIDEr-D        :", f'{metrics["CIDEr-D"]:.6f}')
print("Quality gate              : PASSED")
print("Direct Logit KD           : PROHIBITED")
print("Sequence KD               : REQUIRED / AUTHORIZED")
print("Feature KD                : requires new projection/alignment")
print("Manifest rows             :", len(manifest_rows), "VERIFIED")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("READY marker              :", READY_PATH)
print("READY SHA-256             :", sha256_file(READY_PATH))
print("Progress                  : 7 / 7 code cells")
print(
    "Next                      : Phase 03C ExpansionNet v2 "
    "training/readiness and Train+Restval pseudo-caption contract"
)


PHASE 03B EXPANSIONNET V2 FINAL STATUS
Status                    : PASSED
Teacher                   : A — ExpansionNet v2
Matched Student           : S1 — MobileNetV3-Large
Checkpoint strict load    : PASSED
Validation CIDEr-D        : 1.397207
Quality gate              : PASSED
Direct Logit KD           : PROHIBITED
Sequence KD               : REQUIRED / AUTHORIZED
Feature KD                : requires new projection/alignment
Manifest rows             : 14 VERIFIED
Full training started     : False
Test Set status           : SEALED
READY marker              : /kaggle/working//phase3_teacher_a_expansionnet_v2_preflight/PHASE3_TEACHER_A_EXPANSIONNET_V2_PREFLIGHT_READY.json
READY SHA-256             : ebea12a8c4c37bddd413307a23b166f18b0847830743010037d5e2ef8d98e045
Progress                  : 7 / 7 code cells
Next                      : Phase 03C ExpansionNet v2 training/readiness and Train+Restval pseudo-caption contract
